In [1]:
from __future__ import annotations
from langchain_core.documents import Document
from src.rag.config import RAGConfig
from src.rag.hybrid_retriever import HybridRetriever
from src.rag.hybridrag_langhchain import build_models

d:\Github_ThisPC\muict_thai_legal_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = RAGConfig()
embed_model, reranker, client = build_models(config=config)

retriever = HybridRetriever(
            embed_model=embed_model,
            reranker=reranker,
            client=client,
            config=config,
        )

[RAG] Running on: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<?, ?it/s]


### Retrieve

In [3]:
# query = "มาตราใดจากประมวลกฎหมายแพ่งและพาณิชย์ไม่ถูกนำมาใช้บังคับกับบัตรเงินฝากแม้จะเป็นส่วนหนึ่งของข้อบังคับที่เกี่ยวข้อง"
query = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"

dense_vec, sparse_vec = retriever._encode_query(query)
candidates = retriever._hybrid_search(dense_vec, sparse_vec)
reranked_pts = retriever._rerank(query, candidates)
augmented_context = retriever._link_ref_law(reranked_pts)
# retrieve function
docs = [ # List[Langhchain Docs]
    Document(
        page_content=r.payload.get("text", ""),
        metadata={
            "law_name": r.payload.get("law_name", ""),
            "section_num": r.payload.get("section_num", ""),
            "reference_laws": r.payload.get("reference_laws", []),
            "rank": i,
            "score": r.score,
        },
    )
    for i,r in enumerate(augmented_context, start=1)
]


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [4]:
for doc in docs:
    print(doc)

page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน' metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132', 'reference_laws': [{'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '54'}], 'rank': 1, 'score': 7.8828125}
page_content='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 54 ศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าต้องเป็นนิติบุคคลประเภทบริษัทมหาชนจำกัดและได้รับใบอนุญาตจากคณะกรรมการ ก.ล.ต.
ศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าที่เปิดให้เฉพาะผู้ลงทุนสถาบันซื้อขายสัญญาซื้อขายล่วงหน้าเพื่อตนเองต้องเป็นนิติบุคคลประเภทบริษัทจำกัดหรือบริษัทมหาชนจำกัดและจดทะเบียนกับสำนักงาน ก.ล.ต.
การขออนุญาต การขอจดทะเบียน การออกใบอนุญาต และการรับจดทะเบียนให้เป็นไปตามหลักเกณฑ์ ที่คณะกรรมการ ก.ล.ต. ประ